## Task 1:


In [48]:
import pandas as pd

# to get the list of columns so i can copy and paste them to map them in load.py
df = pd.read_csv("Data_Engineering_Challenge.csv")
df.columns.tolist()

['Site Code',
 'Timestamp',
 'Source Tag',
 'Solar Output Current',
 'Total Load Current',
 'Battery Total Current',
 'Total Voltage']

In [49]:
# checking if lambda function works to strip all space from the columns that have them
df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

df.dtypes

Site Code                    str
Timestamp                    str
Source Tag                   str
Solar Output Current     float64
Total Load Current       float64
Battery Total Current    float64
Total Voltage            float64
dtype: object

In [50]:
# checking for wrong entries
Source_Tag = df["Source Tag"].unique()
print(Source_Tag)

<StringArray>
['Battery             ', 'DG                  ', 'DG+Solar            ',
 'Solar+Battery       ', 'DG+Solar+Battery    ', 'DG+Battery          ']
Length: 6, dtype: str


## Task 2:

Notes:

- The data have 3 min time interval, the challenge demands a 1 hour interval time series data which is aggregated and grouped

- We need to calculate runing hours of all power sources which is going to be:
  `(number of readings of a source in 1 hours * (3 / 60))`

- In many rows of Source Tag the sources are concatinated with a "+" sign So if the substring is present in the main string we have to treat is as "Active" Power Source. and we can use `str.contains()` func to check if the substring is present.

- Output Format is kind of confusing but this is just a dataframe of 1 hour of DG now all of it.


In [51]:
import pandas as pd
import load_data

# loading data using load_data.py
df = load_data.load_dataset("Data_Engineering_Challenge.csv")
df

,Site Code,Timestamp,Source Tag,Solar Output Current,Total Load Current,Battery Total Current,Total Voltage
0,TEST-01,2026-05-09 19:00:00+00:00,Battery,0.0,115.3,-119.3,48.1
1,TEST-01,2026-05-09 19:03:00+00:00,Battery,0.0,115.7,-115.7,48.1
2,TEST-01,2026-05-09 19:06:00+00:00,DG,0.0,109.3,112.1,48.9
3,TEST-01,2026-05-09 19:09:00+00:00,DG,0.0,111.1,114.0,49.1
4,TEST-01,2026-05-09 19:12:00+00:00,DG,0.0,117.1,111.0,49.3
...,...,...,...,...,...,...,...
475,TEST-01,2026-05-10 18:45:00+00:00,Battery,0.0,122.6,-122.2,48.1
476,TEST-01,2026-05-10 18:48:00+00:00,Battery,0.0,110.7,-110.7,48.0
477,TEST-01,2026-05-10 18:51:00+00:00,Battery,0.0,117.7,-112.5,48.0
478,TEST-01,2026-05-10 18:54:00+00:00,Battery,0.0,116.3,-116.3,48.0


In [52]:
df["Timestamp"].dtype

datetime64[us, UTC]

In [53]:
df["Hourly_Timestamp"] = df["Timestamp"].dt.floor("h")

df.head()

,Site Code,Timestamp,Source Tag,Solar Output Current,Total Load Current,Battery Total Current,Total Voltage,Hourly_Timestamp
0,TEST-01,2026-05-09 19:00:00+00:00,Battery,0.0,115.3,-119.3,48.1,2026-05-09 19:00:00+00:00
1,TEST-01,2026-05-09 19:03:00+00:00,Battery,0.0,115.7,-115.7,48.1,2026-05-09 19:00:00+00:00
2,TEST-01,2026-05-09 19:06:00+00:00,DG,0.0,109.3,112.1,48.9,2026-05-09 19:00:00+00:00
3,TEST-01,2026-05-09 19:09:00+00:00,DG,0.0,111.1,114.0,49.1,2026-05-09 19:00:00+00:00
4,TEST-01,2026-05-09 19:12:00+00:00,DG,0.0,117.1,111.0,49.3,2026-05-09 19:00:00+00:00


In [14]:
def running_hours_calculator(df):
    source_tags = ["Battery", "DG", "Solar"]
    rows = []

    for source_tag in source_tags:
        reps = (
            df["Source Tag"].str.contains(source_tag, case=False).sum()
        )  # reps is how many times a source tag appear
        run_hours = (reps * (3 / 60)).round(2)

        # now we check if run_hours are greater than 0 and then append the dict to rows
        if run_hours > 0:
            rows.append(
                {
                    "site code": df["Site Code"].iloc[0],
                    "hour window": df.name,
                    "source": source_tag,
                    "run_hours": run_hours,
                }
            )

    return pd.DataFrame(rows)

In [55]:
# Temporary experiment to see what apply and groupby does
# def running_hours(series):
#     print(type(series))
#     print(series.name)
#     print(series.head())
#     return 0

In [105]:
result = df.groupby("Hourly_Timestamp").apply(running_hours_calculator)
result.reset_index(drop=True, inplace=True)
result

,site code,hour window,source,run_hours
0,TEST-01,2026-05-09 19:00:00+00:00,Battery,0.10
1,TEST-01,2026-05-09 19:00:00+00:00,DG,0.90
2,TEST-01,2026-05-09 20:00:00+00:00,DG,1.00
3,TEST-01,2026-05-09 21:00:00+00:00,DG,1.00
4,TEST-01,2026-05-09 22:00:00+00:00,DG,1.00
5,TEST-01,2026-05-09 23:00:00+00:00,DG,1.00
6,TEST-01,2026-05-10 00:00:00+00:00,DG,1.00
7,TEST-01,2026-05-10 01:00:00+00:00,DG,1.00
8,TEST-01,2026-05-10 01:00:00+00:00,Solar,0.85
9,TEST-01,2026-05-10 02:00:00+00:00,Battery,0.40


In [ ]:
# writing to csv
# result.to_csv("Task2_Output.csv", index=False)

## Task 3:

- we are gonna use same horly grouping as in Task 2.
- we need average kilo-watt (kv) value for each active power source during 1 hour time interval.
- So basically grouping criteria is (site, source, hour)

## Formulas are as follows:

**Diesel Generator (DG) — kW Formula**
No condition check required. Apply to every reading where Source Tag contains "DG".

- `DG kW = Total Load Current × Total Voltage ÷ 1000`

**Battery — kW Formula**
No condition check required. Apply to every reading where Source Tag contains "Battery".

- `Battery kW = Battery Total Current × Total Voltage ÷ 1000`

**Solar — kW Formula**
Apply to every reading where Source Tag contains "Solar". No condition check required.

- `Solar kW = Solar Output Current × Total Voltage ÷ 1000`


In [4]:
import pandas as pd
import load_data

# loading data using load_data.py
df = load_data.load_dataset("Data_Engineering_Challenge.csv")
df

,Site Code,Timestamp,Source Tag,Solar Output Current,Total Load Current,Battery Total Current,Total Voltage
0,TEST-01,2026-05-10 00:00:00+05:00,Battery,0.0,115.3,-119.3,48.1
1,TEST-01,2026-05-10 00:03:00+05:00,Battery,0.0,115.7,-115.7,48.1
2,TEST-01,2026-05-10 00:06:00+05:00,DG,0.0,109.3,112.1,48.9
3,TEST-01,2026-05-10 00:09:00+05:00,DG,0.0,111.1,114.0,49.1
4,TEST-01,2026-05-10 00:12:00+05:00,DG,0.0,117.1,111.0,49.3
...,...,...,...,...,...,...,...
475,TEST-01,2026-05-10 23:45:00+05:00,Battery,0.0,122.6,-122.2,48.1
476,TEST-01,2026-05-10 23:48:00+05:00,Battery,0.0,110.7,-110.7,48.0
477,TEST-01,2026-05-10 23:51:00+05:00,Battery,0.0,117.7,-112.5,48.0
478,TEST-01,2026-05-10 23:54:00+05:00,Battery,0.0,116.3,-116.3,48.0


In [5]:
df["Hourly_Timestamp"] = df["Timestamp"].dt.floor("h")
df.head()

,Site Code,Timestamp,Source Tag,Solar Output Current,Total Load Current,Battery Total Current,Total Voltage,Hourly_Timestamp
0,TEST-01,2026-05-10 00:00:00+05:00,Battery,0.0,115.3,-119.3,48.1,2026-05-10 00:00:00+05:00
1,TEST-01,2026-05-10 00:03:00+05:00,Battery,0.0,115.7,-115.7,48.1,2026-05-10 00:00:00+05:00
2,TEST-01,2026-05-10 00:06:00+05:00,DG,0.0,109.3,112.1,48.9,2026-05-10 00:00:00+05:00
3,TEST-01,2026-05-10 00:09:00+05:00,DG,0.0,111.1,114.0,49.1,2026-05-10 00:00:00+05:00
4,TEST-01,2026-05-10 00:12:00+05:00,DG,0.0,117.1,111.0,49.3,2026-05-10 00:00:00+05:00


In [ ]:
def kilo_watt_calculator(df):
    source_tags = ["Battery", "DG", "Solar"]
    rows = []

    for source_tag in source_tags:
        # as str.contains() return a series of booleans i used it as a mask to get the values for kw calculation
        mask = df["Source Tag"].str.contains(source_tag, case=False)

        # reps is how many times a source tag appear
        reps = df["Source Tag"].str.contains(source_tag, case=False).sum()
        run_hours = (reps * (3 / 60)).round(2)

        # using mask to get the values for formula of kw
        if source_tag == "Battery":
            kw = round(
                (
                    df.loc[mask, "Battery Total Current"]
                    * df.loc[mask, "Total Voltage"]
                    / 1000
                ).mean(),
                2,
            )

        elif source_tag == "DG":
            kw = round(
                (
                    df.loc[mask, "Total Load Current"]
                    * df.loc[mask, "Total Voltage"]
                    / 1000
                ).mean(),
                2,
            )

        elif source_tag == "Solar":
            kw = round(
                (
                    df.loc[mask, "Solar Output Current"]
                    * df.loc[mask, "Total Voltage"]
                    / 1000
                ).mean(),
                2,
            )

        if run_hours > 0:
            rows.append(
                {
                    "site_code": df["Site Code"].iloc[0],
                    "hour_window": df.name,
                    "source": source_tag,
                    "run_hours": run_hours,
                    "kw": kw,
                }
            )

    return pd.DataFrame(rows)

In [12]:
result = df.groupby("Hourly_Timestamp").apply(kilo_watt_calculator)
result.reset_index(drop=True, inplace=True)
result

,site_code,hour_window,source,run_hours,kw
0,TEST-01,2026-05-10 00:00:00+05:00,Battery,0.10,-5.65
1,TEST-01,2026-05-10 00:00:00+05:00,DG,0.90,5.20
2,TEST-01,2026-05-10 01:00:00+05:00,DG,1.00,4.54
3,TEST-01,2026-05-10 02:00:00+05:00,DG,1.00,4.08
4,TEST-01,2026-05-10 03:00:00+05:00,DG,1.00,3.60
5,TEST-01,2026-05-10 04:00:00+05:00,DG,1.00,3.08
6,TEST-01,2026-05-10 05:00:00+05:00,DG,1.00,3.23
7,TEST-01,2026-05-10 06:00:00+05:00,DG,1.00,3.93
8,TEST-01,2026-05-10 06:00:00+05:00,Solar,0.85,0.15
9,TEST-01,2026-05-10 07:00:00+05:00,Battery,0.40,-4.55


In [11]:
# writing the file into csv
result.to_csv("Task3_Output.csv", index=False)

## Creating database


In [8]:
import pandas as pd
from sqlalchemy import create_engine

# loading data using load_data.py
df = pd.read_csv("Data_Engineering_Challenge.csv")
df.head()

,Site Code,Timestamp,Source Tag,Solar Output Current,Total Load Current,Battery Total Current,Total Voltage
0,TEST-01,2026-05-10 00:00:00+05,Battery,0.0,115.3,-119.3,48.1
1,TEST-01,2026-05-10 00:03:00+05,Battery,0.0,115.7,-115.7,48.1
2,TEST-01,2026-05-10 00:06:00+05,DG,0.0,109.3,112.1,48.9
3,TEST-01,2026-05-10 00:09:00+05,DG,0.0,111.1,114.0,49.1
4,TEST-01,2026-05-10 00:12:00+05,DG,0.0,117.1,111.0,49.3


In [9]:
df = df.rename(
    columns={
        "Site Code": "site_code",
        "Source Tag": "source_tag",
        "Timestamp": "timestamp",
        "Solar Output Current": "solar_output_current",
        "Total Load Current": "total_load_current",
        "Battery Total Current": "battery_total_current",
        "Total Voltage": "total_voltage",
    }
)

df.head()

,site_code,timestamp,source_tag,solar_output_current,total_load_current,battery_total_current,total_voltage
0,TEST-01,2026-05-10 00:00:00+05,Battery,0.0,115.3,-119.3,48.1
1,TEST-01,2026-05-10 00:03:00+05,Battery,0.0,115.7,-115.7,48.1
2,TEST-01,2026-05-10 00:06:00+05,DG,0.0,109.3,112.1,48.9
3,TEST-01,2026-05-10 00:09:00+05,DG,0.0,111.1,114.0,49.1
4,TEST-01,2026-05-10 00:12:00+05,DG,0.0,117.1,111.0,49.3


In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

# I always use env variables just to secure my personal info like password
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

# connecting to the database using sqlalchemy engine
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# injecting the data into the database
# i am using append method becasue we already created the table schema as best practice
# "multi" method is used to insert multiple rows instead of indivicual insert statements
df.to_sql(
    "sensor_readings",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
)

480